# Coconut Training Notebook

Modular training pipeline for Coconut (Chain of Continuous Thought) model.  
Supports single-GPU training (no FSDP/DDP required).  
For multi-GPU, use `run.py` with `torchrun`.

## 1. Configuration

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

CONFIG = {
    # Experiment
    "project": "coconut",
    "name": "gsm-coconut-notebook",
    "save_path": "checkpoints",
    "seed": 42,

    # Model
    "model_id": "Qwen/Qwen2.5-1.5B",
    "load_model_path": "None",  # path to checkpoint or "None"
    "bf16": True,
    "sdpa_attention": True,
    "grad_checkpointing": True,

    # Coconut settings
    "coconut": True,
    "cot": False,
    "no_thoughts": False,
    "no_cot": False,
    "c_thought": 2,               # latent tokens per replaced reasoning step
    "epochs_per_stage": 5,        # epochs before advancing curriculum stage
    "max_latent_stage": 3,        # max number of CoT steps replaced by latents
    "pad_latent_to_max": True,
    "termination_gamma": 1.0,     # weight for termination head loss
    "uniform_prob": 0.0,          # probability of random stage sampling per sample

    # Data
    "train_path": "data/gsm_train.json",
    "val_path": "data/gsm_valid.json",

    # Training
    "num_epochs": 25,
    "resume": 0,                  # skip first N epochs (set when loading checkpoint)
    "batch_size_training": 1,
    "gradient_accumulation_steps": 32,
    "lr": 1e-4,
    "weight_decay": 0.01,
    "reset_optimizer": False,     # reset optimizer each epoch

    # Eval & saving
    "only_eval": False,
    "save_only_improve": True,
    "debug": False,
}

## 2. Imports & Setup

In [ ]:
import gc
import json
import datetime
import itertools

import torch
import bitsandbytes as bnb
import wandb
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

from coconut import Coconut
from dataset import (
    get_dataset,
    get_question_latent_dataset,
    get_cot_latent_dataset,
    MyCollator,
)
from utils import Config, set_seed

In [ ]:
configs = Config(CONFIG)
set_seed(configs.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

save_dir = os.path.join(configs.save_path, configs.name)
os.makedirs(save_dir, exist_ok=True)
current_time = datetime.datetime.now().strftime("%y%m%d_%H%M%S")

## 3. Model & Tokenizer

In [ ]:
# Load base model
load_kwargs = {}
if configs.sdpa_attention:
    load_kwargs["attn_implementation"] = "sdpa"
if configs.bf16:
    load_kwargs["torch_dtype"] = torch.bfloat16

base_model = AutoModelForCausalLM.from_pretrained(configs.model_id, **load_kwargs)

if configs.grad_checkpointing:
    base_model.gradient_checkpointing_enable()

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(configs.model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.add_tokens("<|start-latent|>")
tokenizer.add_tokens("<|end-latent|>")
tokenizer.add_tokens("<|latent|>")

latent_id = tokenizer.convert_tokens_to_ids("<|latent|>")
start_id = tokenizer.convert_tokens_to_ids("<|start-latent|>")
end_id = tokenizer.convert_tokens_to_ids("<|end-latent|>")

print(f"Special token IDs - latent: {latent_id}, start: {start_id}, end: {end_id}")

In [ ]:
# Initialize new token embeddings
if not (configs.cot or configs.no_thoughts or configs.no_cot):
    base_model.resize_token_embeddings(len(tokenizer))
    embeddings = base_model.get_input_embeddings()
    target_id = tokenizer.convert_tokens_to_ids("<<")
    for token_id in [latent_id, start_id, end_id]:
        target_embedding = embeddings.weight.data[target_id]
        embeddings.weight.data[token_id] = target_embedding
        lm_head = base_model.lm_head
        lm_head.weight.data[token_id] = lm_head.weight.data[target_id]
    print("Initialized special token embeddings from '<<' token")

# Handle no_thoughts mode
if configs.no_thoughts:
    configs.c_thought = 0
    configs.coconut = False

# Wrap in Coconut if needed
if configs.coconut:
    model = Coconut(base_model, latent_id, start_id, end_id, tokenizer.eos_token_id, configs.termination_gamma)
    print("Wrapped model in Coconut")
else:
    model = base_model
    print("Using base model (no Coconut)")

# Load checkpoint if specified
if configs.load_model_path != "None":
    saved_weights = torch.load(configs.load_model_path, map_location=device)
    result = model.load_state_dict(saved_weights, strict=False)
    print(f"Loaded checkpoint: {configs.load_model_path}")
    if result.missing_keys:
        print(f"  Missing keys: {len(result.missing_keys)}")
    if result.unexpected_keys:
        print(f"  Unexpected keys: {len(result.unexpected_keys)}")

if configs.bf16:
    model = model.to(torch.bfloat16)

model = model.to(device)

n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters: {n_params:,} total, {n_trainable:,} trainable")

## 4. Datasets

In [ ]:
# Load base datasets (tokenized, stage-independent)
max_data = 3200 if configs.debug else 10**9

base_dataset_train = get_dataset(configs.train_path, tokenizer, max_size=max_data)
base_dataset_valid = get_dataset(configs.val_path, tokenizer, max_size=32 if configs.debug else 10**9)

# Load validation ground truth
val_data = json.load(open(configs.val_path))
question_val = [d["question"] for d in val_data]
answers_val = [d["answer"].replace(",", "").strip() for d in val_data]
cot_val = ["\n".join(d["steps"]) for d in val_data]

max_new_tokens = 64 if "gsm" in configs.val_path else 128

collator = MyCollator(tokenizer, latent_id=latent_id, label_pad_token_id=-100)

print(f"Train: {len(base_dataset_train)} samples, Valid: {len(base_dataset_valid)} samples")
print(f"Max new tokens for eval: {max_new_tokens}")

## 5. Helper Functions

In [ ]:
def print_memory_breakdown(model, optimizer):
    """Print VRAM usage breakdown."""
    param_mem = sum(p.numel() * p.element_size() for p in model.parameters())
    grad_mem = sum(p.grad.numel() * p.grad.element_size() for p in model.parameters() if p.grad is not None)
    opt_mem = sum(
        v.numel() * v.element_size()
        for state in optimizer.state.values()
        for k, v in state.items() if torch.is_tensor(v)
    )
    total_allocated = torch.cuda.memory_allocated()
    to_gb = 1024**3
    print(f"--- VRAM Breakdown ---")
    print(f"Model Weights:    {param_mem / to_gb:.2f} GB")
    print(f"Gradients:        {grad_mem / to_gb:.2f} GB")
    print(f"Optimizer States: {opt_mem / to_gb:.2f} GB")
    print(f"Activations/Misc: {(total_allocated - param_mem - grad_mem - opt_mem) / to_gb:.2f} GB")
    print(f"Total Allocated:  {total_allocated / to_gb:.2f} GB")
    print(f"Max Allocated:    {torch.cuda.max_memory_allocated() / to_gb:.2f} GB")


def build_dataloaders(scheduled_stage):
    """Build train, val-loss, and val-gen dataloaders for a given curriculum stage."""
    no_special = configs.cot or configs.no_cot or configs.no_thoughts

    # Training data (lazy, stochastic)
    dataset_train = get_cot_latent_dataset(
        scheduled_stage, base_dataset_train, configs,
        start_id, latent_id, end_id,
        no_special_marker=no_special, shuffle=True,
    )
    train_loader = torch.utils.data.DataLoader(
        dataset_train, batch_size=configs.batch_size_training,
        shuffle=True, num_workers=1, pin_memory=True, collate_fn=collator,
    )

    # Validation loss data
    dataset_val_loss = get_cot_latent_dataset(
        scheduled_stage, base_dataset_valid, configs,
        start_id, latent_id, end_id,
        no_special_marker=no_special,
    )
    val_loss_loader = torch.utils.data.DataLoader(
        dataset_val_loss, batch_size=configs.batch_size_training,
        shuffle=False, num_workers=1, pin_memory=True, collate_fn=collator,
    )

    # Validation generation data
    dataset_val_gen = get_question_latent_dataset(
        scheduled_stage, base_dataset_valid, configs,
        start_id, latent_id, end_id,
        no_special_marker=no_special,
    )
    val_gen_loader = torch.utils.data.DataLoader(
        dataset_val_gen, batch_size=1,
        shuffle=False, num_workers=1, pin_memory=True, collate_fn=collator,
    )

    return train_loader, val_loss_loader, val_gen_loader


def save_checkpoint(model, optimizer, epoch, save_dir, suffix=""):
    """Save model and optimizer state."""
    path = os.path.join(save_dir, f"checkpoint_{epoch + 1}{suffix}")
    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
    }
    torch.save(checkpoint, path)
    print(f"Saved checkpoint: {path}")
    return path

## 6. Training, Validation, and Evaluation Functions

In [ ]:
def train_one_epoch(model, train_loader, optimizer, epoch, wandb_run=None):
    """Run one training epoch. Returns average loss."""
    model.train() if not configs.coconut else model.module.train() if hasattr(model, 'module') else model.train()

    total_loss = 0.0
    n_steps = 0
    accum_steps = configs.gradient_accumulation_steps
    total_opt_steps = len(train_loader) // accum_steps

    pbar = tqdm(train_loader, desc=f"Train Epoch {epoch+1}", dynamic_ncols=True)

    for step, batch in enumerate(pbar):
        batch = {k: v.to(device) for k, v in batch.items() if k != "idx"}

        outputs = model(**batch)
        loss = outputs.loss / accum_steps
        loss.backward()

        step_loss = loss.item() * accum_steps
        total_loss += step_loss
        n_steps += 1

        if (step + 1) % accum_steps == 0 or step == len(train_loader) - 1:
            optimizer.step()
            optimizer.zero_grad()

        pbar.set_postfix(loss=f"{step_loss:.4f}")

        if wandb_run:
            log_dict = {
                "train/epoch": epoch + 1,
                "train/step": epoch * len(train_loader) + step,
                "train/loss": step_loss,
            }
            if hasattr(outputs, 'termination_logits') and outputs.termination_logits is not None:
                log_dict["train/term_acc"] = (
                    torch.sum(torch.argmax(outputs.termination_logits, dim=-1) == outputs.termination_labels).item()
                    / (outputs.logits.shape[0] * outputs.logits.shape[1])
                )
            wandb_run.log(log_dict)

        if step == 100:
            print_memory_breakdown(model, optimizer)

    avg_loss = total_loss / n_steps if n_steps > 0 else 0.0
    print(f"  Train loss: {avg_loss:.4f}")
    return avg_loss

In [ ]:
def validate_loss(model, val_loss_loader):
    """Compute validation loss. Returns average loss."""
    model.eval() if not configs.coconut else model.module.eval() if hasattr(model, 'module') else model.eval()

    total_loss = 0.0
    n_batches = 0

    with torch.no_grad():
        for batch in tqdm(val_loss_loader, desc="Val Loss", leave=False):
            batch = {k: v.to(device) for k, v in batch.items() if k != "idx"}
            outputs = model(**batch)
            total_loss += outputs.loss.item()
            n_batches += 1

    avg_loss = total_loss / n_batches if n_batches > 0 else 0.0
    print(f"  Val loss: {avg_loss:.4f}")
    return avg_loss

In [ ]:
def evaluate_generation(model, val_gen_loader, epoch, save_dir):
    """Run generation evaluation. Returns (accuracy, cot_match_rate)."""
    model.eval() if not configs.coconut else model.module.eval() if hasattr(model, 'module') else model.eval()

    # Access the underlying model for .generate()
    gen_model = model.module if hasattr(model, 'module') else model

    cor = 0
    cor_cot = 0
    total = 0
    eval_path = os.path.join(save_dir, f"checkpoint_{epoch + 1}_evaluation_{current_time}.txt")

    with torch.no_grad():
        pbar = tqdm(val_gen_loader, desc="Eval Gen", dynamic_ncols=True)

        for idx, batch in enumerate(pbar):
            test_idx = batch["idx"][0]

            batch = {
                k: v.to(device)
                for k, v in batch.items()
                if v is not None and k not in ["idx", "position_ids"]
            }

            assert len(batch["input_ids"]) == 1
            answer = answers_val[test_idx.cpu().item()]
            answer_cot = cot_val[test_idx.cpu().item()]

            total += 1

            outputs = gen_model.generate(
                **batch,
                max_new_tokens=max_new_tokens,
                synced_gpus=False,
            )

            text_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
            answer_output = text_output.split("#")[-1].replace(",", "").strip()
            cot_output = ("\n".join(text_output.split("\n")[1:])).split("#")[0].strip()

            if idx < 5:
                print(f"  Q{test_idx}: answer='{answer}' | output='{answer_output}'")
                print(f"    Full: '{tokenizer.decode(outputs[0])}'")

            cor += (answer_output == answer)
            cor_cot += (cot_output == answer_cot)

            pbar.set_postfix(acc=f"{cor/total:.2%}")

            with open(eval_path, "a+") as fp:
                fp.write(f"\nQuestion {test_idx+1}:\nCoT = {answer_cot}\n")
                fp.write(f"Full output:\n{tokenizer.decode(outputs[0])}\n")
                fp.write(f"Extracted Output:\n{answer_output}\n")
                fp.write(f"Answer = {answer}\n")
                fp.write("-" * 30)

    acc = cor / total if total > 0 else 0.0
    cot_em = cor_cot / total if total > 0 else 0.0
    print(f"  Accuracy: {cor}/{total} = {acc:.2%}")
    print(f"  CoT EM:   {cor_cot}/{total} = {cot_em:.2%}")
    return acc, cot_em

## 7. Training Loop

In [ ]:
# Initialize wandb
wandb_run = None
if not configs.debug and not configs.only_eval:
    wandb_run = wandb.init(project=configs.project, name=configs.name)
    wandb_run.config.update(CONFIG, allow_val_change=True)

In [ ]:
# Initialize optimizer
optimizer = bnb.optim.Adam8bit(
    model.parameters(),
    lr=configs.lr,
    weight_decay=configs.weight_decay,
)

best_acc = 0.0
prev_stage = -1

In [ ]:
for epoch in range(configs.resume, configs.num_epochs):
    # Curriculum stage
    scheduled_stage = (
        0 if (configs.cot or configs.no_cot) else epoch // configs.epochs_per_stage
    )
    print(f"\n{'='*60}")
    print(f"Epoch {epoch+1}/{configs.num_epochs} | Stage {scheduled_stage}")
    print(f"{'='*60}")

    # Reset optimizer at stage transitions
    if configs.reset_optimizer or (scheduled_stage != prev_stage and prev_stage >= 0):
        del optimizer
        optimizer = bnb.optim.Adam8bit(
            model.parameters(),
            lr=configs.lr,
            weight_decay=configs.weight_decay,
        )
        if scheduled_stage != prev_stage:
            print(f"  Optimizer reset (stage transition {prev_stage} -> {scheduled_stage})")
    prev_stage = scheduled_stage

    # Build dataloaders for current stage
    train_loader, val_loss_loader, val_gen_loader = build_dataloaders(scheduled_stage)

    # --- Train ---
    if not configs.only_eval:
        train_loss = train_one_epoch(model, train_loader, optimizer, epoch, wandb_run)

        # Save checkpoint (every epoch if not save_only_improve)
        if not configs.save_only_improve and not configs.debug:
            save_checkpoint(model, optimizer, epoch, save_dir)
            gc.collect()
            torch.cuda.empty_cache()

        # --- Validate loss ---
        val_loss = validate_loss(model, val_loss_loader)
        if wandb_run:
            wandb_run.log({"val/loss": val_loss})

    # --- Evaluate generation ---
    acc, cot_em = evaluate_generation(model, val_gen_loader, epoch, save_dir)
    if wandb_run:
        wandb_run.log({"eval/acc": acc, "eval/cot_em": cot_em})

    if configs.only_eval:
        break

    # Save best model
    if acc > best_acc and configs.save_only_improve and not configs.debug:
        save_checkpoint(model, optimizer, epoch, save_dir, suffix=f"_{current_time}.pt")
        best_acc = acc
        gc.collect()
        torch.cuda.empty_cache()

    # Cleanup
    del train_loader, val_loss_loader, val_gen_loader
    gc.collect()

print(f"\nTraining complete. Best accuracy: {best_acc:.2%}")

In [ ]:
if wandb_run:
    wandb_run.finish()